# T59 — Debugging a plate model: negative divergence at ridges, negative convergence at trenches, and transform velocities

**Cluster M: Plate-model debugging** (1 / 4).

## What this notebook does

Plate models encode a huge amount of geometry: mid-ocean ridges (MORs), subduction zones (SZs), and transforms are drawn as line features and expected to move in specific ways when the rotation model is applied. In a well-constructed model, MOR sub-segments should DIVERGE, SZ sub-segments should CONVERGE, and transform sub-segments should have negligible orthogonal (across-boundary) velocity. When any of those expectations is violated for a sub-segment, it flags a **construction anomaly** — either the topology is wrong (e.g. a line drawn as a ridge but functioning as a trench), or the rotations aren't consistent with the topology, or a plate-ID is missing.

This notebook walks through the diagnostic pattern used to find those anomalies:

1. **§1** Load the plate model (Zahirovic 2022 by default) and the diagnostic helper `diagnose_topology_convergence` from the bundled `plate_model_debug` package.
2. **§2** For each snapshot in **100, 125, 150, 175, 200 Ma**, compute the orthogonal convergence velocity at every MOR / SZ / transform sub-segment midpoint.
3. **§3** Render one pyGMT map per snapshot highlighting:
   - MOR sub-segments with **positive** orthogonal convergence velocity (i.e. **negative divergence**, they should be spreading but aren't).
   - SZ sub-segments with **negative** orthogonal convergence velocity (i.e. **negative convergence**, they should be closing but aren't).
   - Transform sub-segments with any non-zero orthogonal convergence velocity (transforms should be strike-slip = orthogonal component ~ 0).
4. **§4** Build a **compact MP4 video** at 1-Myr cadence over the full 100-200 Ma window so the user can watch anomalies appear, migrate, and disappear as topologies change through time.
5. **§5** Extend this — swap in a different plate model, extend the time window, run over 0-1000 Ma, adjust the velocity thresholds.

**Reading the anomalies.** No plate model is perfect — every published model carries some residual number of these flags because fixing them all is time-consuming manual work. The point of the diagnostic is NOT that a clean model has zero anomalies (usually impossible) but that you can quickly SEE where the anomalies are, decide which matter for your downstream analysis, and either fix them or work around them.

**Audience**: intermediate → researcher (basic pyGPlates familiarity assumed).
**Difficulty**: ★★★.
**Runtime**: ~1 min for the 5 static snapshots; ~5-10 min for the 1-Myr video (101 frames).


## Data availability

**Plate model** — Zahirovic 2022 (`Zahirovic2022`), fetched via the Plate Model Manager on first run. This is the suite default. Any PMM model can be swapped in via `MODEL_NAME` in the USER CONFIGURATION block.

**Helper module** — `Notebooks/plate_model_debug/` (bundled with the suite). Wraps upstream code from the private repo <https://github.com/EarthByte/plate-model-debug> (Ben Sculley + John Cannon, EarthByte), being made public via this tutorial suite.

**No CSV / NetCDF inputs required** — everything derives from the plate model's rotation file + topology GPMLs.

## Sources

- Sculley, B. & Cannon, J. (2025) *plate-model-debug*. EarthByte, University of Sydney. Companion code for these tutorials.
- Zahirovic, S. et al. (2022) A 1.8 billion year history of the plate tectonic evolution of Earth. *Geoscience Frontiers* (in review) — reference for the plate model used as the default here.


## Environment + imports


In [ ]:
from pathlib import Path
import os, sys, warnings, tempfile, shutil
if Path("../data").exists() and not Path("data").exists():
    os.chdir("..")

import numpy as np
import matplotlib.pyplot as plt
import gplately
import pygmt
import pygplates
from plate_model_manager import PlateModelManager

# Bundled helper module
sys.path.insert(0, str(Path("Notebooks").resolve()))
from plate_model_debug import diagnose_topology_convergence

print("Environment")
print(f"  python      {sys.version.split()[0]}")
for _m in (np, gplately, pygmt, pygplates):
    print(f"  {_m.__name__:11s} {getattr(_m, '__version__', 'n/a')}")


In [ ]:
# === USER CONFIGURATION =====================================================
# Plate model + anchor (Z22 is the tutorial-suite default).
MODEL_NAME             = "Zahirovic2022"
ANCHOR_PLATE_ID        = 0

# Static snapshot ages for the §3 pyGMT panels.
SNAPSHOT_AGES_MA       = [100, 125, 150, 175, 200]

# Video: 1-Myr cadence over 100-200 Ma (compact, ~5 MB).
VIDEO_START_MA         = 200        # oldest frame
VIDEO_END_MA           = 100        # youngest frame
VIDEO_CADENCE_MA       = 1          # frame every 1 Myr
VIDEO_FPS              = 10         # playback speed
VIDEO_WIDTH_PX         = 800        # frame width in pixels (compact)
VIDEO_CRF              = 28         # h264 quality (23=default, 28=compact)

# Velocity delta-time and sampling for diagnose_topology_convergence
# (Ben's defaults; smaller sampling = denser, slower).
VELOCITY_DELTA_TIME_MYR = 1.0
SAMPLING_DISTANCE_KM    = 100.0     # ~ 100 km along-boundary spacing

# Map region + projection.
REGION_GLOBAL          = [-180, 180, -75, 80]
PROJECTION             = "N15c"     # Robinson, 15 cm wide

# Output dirs (gitignored per suite convention).
FRAMES_DIR             = Path("Notebooks/T59_debug_divergence_frames")
VIDEO_DIR              = Path("Notebooks/videos")
VIDEO_PATH             = VIDEO_DIR / "T59_debug_divergence.mp4"
# ============================================================================
print(f"  plate model:   {MODEL_NAME}")
print(f"  snapshots:     {SNAPSHOT_AGES_MA} Ma")
print(f"  video window:  {VIDEO_END_MA}-{VIDEO_START_MA} Ma at {VIDEO_CADENCE_MA}-Myr cadence")
print(f"  video output:  {VIDEO_PATH}  ({VIDEO_WIDTH_PX}px wide, crf {VIDEO_CRF})")


## 1. Load plate model + helper


In [ ]:
pmm = PlateModelManager()
model = pmm.get_model(MODEL_NAME, data_dir="./gplately_data")

# PMM may return the rotation model as a list of paths; wrap if needed
_pmm_rot = model.get_rotation_model()
if not hasattr(_pmm_rot, "get_rotation"):
    rotation_model = pygplates.RotationModel(_pmm_rot)
else:
    rotation_model = _pmm_rot

# Load the topology features (feature-collection form, needed by resolve_topologies)
topology_features = [pygplates.FeatureCollection(f) for f in model.get_topologies()]

recon = gplately.PlateReconstruction(
    rotation_model=rotation_model,
    topology_features=model.get_topologies(),
    static_polygons=model.get_static_polygons(),
    anchor_plate_id=ANCHOR_PLATE_ID,
)
print(f"  plate model loaded: {MODEL_NAME}")
print(f"  topology feature collections: {len(topology_features)}")


## 2. Compute anomalies per snapshot

For each snapshot age, three calls to `diagnose_topology_convergence` — one filtered to MOR features (threshold: only return positive-convergence sub-segments = negative divergence), one filtered to SZ features (threshold: only return negative-convergence sub-segments = negative convergence), one filtered to transforms (no threshold — return all).


In [ ]:
def diagnose_snapshot(time):
    """Return (mor_anomalies, sz_anomalies, transform_all) for one age snapshot."""
    # MORs — return sub-segments with POSITIVE convergence (i.e. wrong-sign)
    mor_features, mor_vels = diagnose_topology_convergence(
        rotation_model, topology_features, time,
        convergent_velocity_threshold_cms_yr=0,
        boundary_feature_types=[pygplates.FeatureType.gpml_mid_ocean_ridge],
        velocity_delta_time=VELOCITY_DELTA_TIME_MYR,
        threshold_sampling_distance_radians=SAMPLING_DISTANCE_KM / pygplates.Earth.mean_radius_in_kms,
        anchor_plate_id=ANCHOR_PLATE_ID,
    )
    # SZs — return sub-segments with NEGATIVE convergence (i.e. wrong-sign)
    sz_features, sz_vels = diagnose_topology_convergence(
        rotation_model, topology_features, time,
        divergent_velocity_threshold_cms_yr=0,
        boundary_feature_types=[pygplates.FeatureType.gpml_subduction_zone],
        velocity_delta_time=VELOCITY_DELTA_TIME_MYR,
        threshold_sampling_distance_radians=SAMPLING_DISTANCE_KM / pygplates.Earth.mean_radius_in_kms,
        anchor_plate_id=ANCHOR_PLATE_ID,
    )
    # Transforms — return everything (no threshold)
    tr_features, tr_vels = diagnose_topology_convergence(
        rotation_model, topology_features, time,
        boundary_feature_types=[pygplates.FeatureType.gpml_transform],
        velocity_delta_time=VELOCITY_DELTA_TIME_MYR,
        threshold_sampling_distance_radians=SAMPLING_DISTANCE_KM / pygplates.Earth.mean_radius_in_kms,
        anchor_plate_id=ANCHOR_PLATE_ID,
    )
    return (mor_features, mor_vels), (sz_features, sz_vels), (tr_features, tr_vels)

# Quick sanity call at 150 Ma
_mor, _sz, _tr = diagnose_snapshot(150.0)
print(f"  150 Ma:  {len(_mor[0])} MOR anomalies, {len(_sz[0])} SZ anomalies, "
      f"{len(_tr[0])} transform sub-segments (all shown)")


## 3. Static snapshots at 100, 125, 150, 175, 200 Ma

One pyGMT map per snapshot. Colour coding:

- **Cyan lines** — MOR sub-segments with negative divergence (topology or rotation problem)
- **Magenta lines** — SZ sub-segments with negative convergence
- **Yellow lines** — transform sub-segments with any non-zero orthogonal velocity (colour intensity = magnitude)

Base map: coastlines + all plate-boundary topological sections (the continuous-backbone pattern) in grey, so the anomalous sub-segments stand out on top.


In [ ]:
def render_snapshot_map(time, out_path=None, width_cm=15):
    """Render one snapshot; write to `out_path` if given, else return the figure."""
    (mor_feats, _), (sz_feats, _), (tr_feats, tr_vels) = diagnose_snapshot(time)

    gplot = gplately.PlotTopologies(
        plate_reconstruction=recon,
        coastlines=model.get_coastlines(),
        continents=model.get_continental_polygons(),
        COBs=model.get_COBs(),
        time=float(time),
        plot_engine=gplately.PygmtPlotEngine(),
    )

    fig = pygmt.Figure()
    fig.basemap(region=REGION_GLOBAL, projection=f"N{width_cm}c", frame=["af", "WSne"])

    # Continents in light grey (house style)
    try:
        gplot.plot_continents(fig, fill="gray95", pen="0.2p,gray40")
        gplot.plot_coastlines(fig, pen="0.3p,gray20")
    except Exception:
        pass

    # Continuous backbone of all topological sections in mid-grey
    try:
        engine = gplot._plot_engine if hasattr(gplot, "_plot_engine") else gplately.PygmtPlotEngine()
        engine.plot_geo_data_frame(fig, gplot.get_all_topological_sections(),
                                    pen="0.4p,gray60")
    except Exception:
        pass

    # Anomalous lines — extract polyline geometries and draw them
    def _plot_features(features, pen):
        for f in features:
            for geom in f.get_geometries():
                pts = geom.to_lat_lon_array()
                fig.plot(x=pts[:, 1], y=pts[:, 0], pen=pen)

    _plot_features(mor_feats, pen="1.6p,cyan")
    _plot_features(sz_feats,  pen="1.6p,magenta")
    _plot_features(tr_feats,  pen="0.8p,yellow2")

    fig.text(text=f"{int(time)} Ma  ({MODEL_NAME})  "
                   f"MOR anomalies: {len(mor_feats)}  |  "
                   f"SZ anomalies: {len(sz_feats)}  |  "
                   f"transform sub-segments: {len(tr_feats)}",
              position="TL", offset="0.25c/-0.25c", justify="TL",
              font="10p,Helvetica-Bold,black", fill="white", pen="0.5p,gray40")

    if out_path is not None:
        fig.savefig(str(out_path), dpi=100)
        return None
    return fig

# Render all 5 static snapshots
for _age in SNAPSHOT_AGES_MA:
    print(f"  rendering {_age} Ma snapshot ...")
    _f = render_snapshot_map(_age, width_cm=15)
    _f.show(width=900)


### How to read these maps

- **Cyan segments** on any ridge line = the two plates on either side of that segment are moving TOWARD each other in this model. Either the ridge geometry is drawn incorrectly (should be a trench or a transform), or the rotation model has the wrong sense of motion for the plates involved, or the plate-ID assignment is wrong.
- **Magenta segments** on any trench line = the two plates on either side are moving APART. Same three possible causes, but flipped.
- **Yellow segments** on any transform = the transform has a non-zero across-boundary velocity component. In a well-drawn model, transforms are pure strike-slip (velocity along the boundary, zero across it). Non-zero across-boundary velocity means the "transform" is functioning partly as a ridge or a trench — usually a legacy of an active-margin transition that was drawn as a transform but hasn't yet been re-typed.

**Watch for**

- **Snapshot 200 Ma** — early Mesozoic Panthalassa, lots of subduction, likely to show the most SZ anomalies where absolute-motion reconstructions of the Panthalassic plate are least constrained.
- **Snapshot 100 Ma** — mid-Cretaceous, well-constrained by preserved seafloor. Anomaly count should be at its lowest here.
- **Anomaly location, not just count** — a single cyan segment at the North Atlantic MOR is a different fix than a cluster of cyan segments along the entire Pacific-Farallon boundary.


## 4. Compact MP4 video at 1-Myr cadence over 100-200 Ma

Same rendering as §3, but for every 1-Myr step in the window. Frames are stitched with ffmpeg (via `imageio-ffmpeg`, which ships with a bundled binary — no external install needed). Video is written to `Notebooks/videos/T59_debug_divergence.mp4` (gitignored) and displayed inline via HTML5.

Runtime: ~5-10 minutes on a laptop for 101 frames at compact settings.


In [ ]:
# Build MP4 by rendering per-frame PNGs and stitching with imageio-ffmpeg.

FRAMES_DIR.mkdir(exist_ok=True, parents=True)
VIDEO_DIR.mkdir(exist_ok=True, parents=True)

# Frame width in cm (pygmt) computed from target pixel width at ~100 dpi
_target_dpi = 100
_frame_width_cm = VIDEO_WIDTH_PX / _target_dpi * 2.54

ages = np.arange(VIDEO_END_MA, VIDEO_START_MA + 1e-6, VIDEO_CADENCE_MA)
ages_desc = ages[::-1]  # oldest first for a "watch time run forward" playback

print(f"  will render {len(ages_desc)} frames at {_frame_width_cm:.1f} cm wide "
      f"(~{VIDEO_WIDTH_PX} px)")
print(f"  frame directory: {FRAMES_DIR}")

# Render frames (skip if already on disk — makes reruns cheap)
frame_paths = []
for i, _age in enumerate(ages_desc):
    _p = FRAMES_DIR / f"frame_{int(round(_age)):04d}Ma.png"
    if not _p.exists():
        render_snapshot_map(float(_age), out_path=_p, width_cm=_frame_width_cm)
    if i % 10 == 0:
        print(f"    frame {i+1}/{len(ages_desc)}  ({int(round(_age))} Ma)")
    frame_paths.append(_p)
print(f"  ✓ {len(frame_paths)} frames on disk")

# Encode with imageio-ffmpeg
import imageio.v2 as imageio
import imageio_ffmpeg

writer = imageio.get_writer(
    str(VIDEO_PATH), fps=VIDEO_FPS, codec="libx264",
    quality=None, ffmpeg_params=["-crf", str(VIDEO_CRF),
                                    "-preset", "veryfast",
                                    "-pix_fmt", "yuv420p"],
)
for _p in frame_paths:
    writer.append_data(imageio.imread(str(_p)))
writer.close()
print(f"  ✓ wrote {VIDEO_PATH}  ({VIDEO_PATH.stat().st_size / 1e6:.1f} MB)")

# Display inline
from IPython.display import Video
Video(str(VIDEO_PATH), embed=True, width=VIDEO_WIDTH_PX)


## Extend this

- **Different plate model** — swap `MODEL_NAME` to `"Muller2019"`, `"Muller2022"`, `"Cao2024"`, `"Merdith2021"`, or any PMM model listed at <https://repo.gplates.org/webdav/pmm/config/models_v2.json>. Each will have a different anomaly pattern — comparing them side-by-side is a fast way to see which regions are well-constrained across models vs disputed.
- **Different time window** — bump `VIDEO_START_MA` / `VIDEO_END_MA` to cover 0-1000 Ma. Runtime scales linearly with frames; a 1000-frame video takes ~1 hour at these compact settings.
- **Your own plate model** — point `pygplates.RotationModel` at your rotation file and `pygplates.FeatureCollection` at your topology GPMLs, skip the `plate_model_manager` fetch, and re-run everything downstream. The helper module makes no assumption about which model you're using.
- **Tighter velocity thresholds** — set `convergent_velocity_threshold_cms_yr=0.5` (say) on the MOR call to only flag sub-segments with STRONGLY wrong-sign convergence (> 0.5 cm/yr toward each other). Filters out numerical noise near stationary plates.
- **Zoom into a specific region** — swap `REGION_GLOBAL` for a regional bounding box (e.g. `[-180, -60, -60, 15]` for the SE Pacific + South America) and change projection to Mercator (`M15c`) for high-resolution local diagnostics.

## Related resources

- Upstream repo: <https://github.com/EarthByte/plate-model-debug> — made public via this suite.
- Companion notebooks in this cluster: **T60** velocity magnitude at MORs, **T61** topology construction anomalies (gaps/overlaps + non-unique sections + missing polarity), **T62** feature extractability at subduction zones.

## References

- Zahirovic, S., Cannon, J., Chin, M., Müller, R.D. (2022) A 1.8 billion year history of the plate tectonic evolution of Earth. *Geoscience Frontiers* (in review) — the default plate model here.
- Müller, R.D., et al. (2019) A global plate model including lithospheric deformation along major rifts and orogens since the Triassic. *Tectonics* 38(6), 1884-1907. — alternative plate model.
- Mather, B.R., et al. (2024) Deep time spatio-temporal data analysis using pyGPlates with PlateTectonicTools and GPlately. *Applied Computing and Geosciences* 22, 100152.
